# Embeddings of images using OpenCV

In [29]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
import numpy as np
import cv2
from torch.utils.data import DataLoader, Dataset
from PIL import Image

In [3]:
def initialize_resnet_model(model_name='resnet18'):
    """
    Initialize a ResNet model for feature extraction.

    Args:
        model_name (str): The name of the ResNet model ('resnet18' or 'resnet50').

    Returns:
        nn.Module: The ResNet model.
    """
    # Load the ResNet model and move it to the GPU if available
    if model_name == 'resnet18':
        model = models.resnet18()
    elif model_name == 'resnet50':
        model = models.resnet50()
    else:
        raise ValueError("Unsupported model name. Supported options are 'resnet18' and 'resnet50'.")

    # Remove the final classification layer
    model = nn.Sequential(*list(model.children())[:-1])

    # Move the model to the GPU if available
    if torch.cuda.is_available():
        model = model.to('cuda')

    # Set the model to evaluation mode
    model.eval()

    return model

In [4]:
model = initialize_resnet_model()

In [5]:
class CustomDataset(Dataset):
    def __init__(self, data, target_size=(640, 640)):
        """
        Custom dataset for loading and preprocessing images.

        Args:
            data (list): List of image data, where each element can be either a file path or an OpenCV numpy array.
            target_size (tuple): Target size for resizing and padding the images.
        """
        self.data = data
        self.target_size = target_size
        self.transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        data_item = self.data[idx]

        if isinstance(data_item, str):
            img = cv2.imread(data_item)
            height, width, _ = img.shape
            if width < self.target_size[0] or height < self.target_size[1]:
                padding_width = max(self.target_size[0] - width, 0)
                padding_height = max(self.target_size[1] - height, 0)
                img = cv2.copyMakeBorder(img, 0, padding_height, 0, padding_width, cv2.BORDER_CONSTANT, value=(255, 255, 255))
            img = cv2.resize(img, self.target_size)
            img = self.transform(img)
        elif isinstance(data_item, np.ndarray):
            img = cv2.resize(data_item, self.target_size)
            img = self.transform(img)
        else:
            raise ValueError("Unsupported data type. Data must be a file path or an OpenCV numpy array.")
        
        # Move the image to the GPU if available
        if torch.cuda.is_available():
            img = img.to('cuda')

        return img


In [10]:
def extract_resnet_features(data_loader, model):
    """
    Extract ResNet feature embeddings for a list of image crops with batching.

    Args:
        data_loader (DataLoader): DataLoader for image batches.
        model (nn.Module): Pre-initialized ResNet model.
        batch_size (int): Batch size for processing.

    Returns:
        numpy.ndarray: Feature embeddings for the image crops.
    """
    features_list = []

    with torch.no_grad():
        for batch in data_loader:
            features = model(batch)
            features = features.view(features.size(0), -1)
            features /= features.norm(p=2, dim=1, keepdim=True)
            features_list.extend(features.detach().cpu().numpy())

    return np.vstack(features_list)

In [6]:
# Initialize the ResNet model once
resnet_model = initialize_resnet_model(model_name='resnet18')

In [7]:
img_path = '/mnt/nas/back_ubuntu/superdave/Desktop/killboy_sm_ai/img_0393.jpg'
img_path2 = '/mnt/nas/back_ubuntu/superdave/Desktop/killboy_sm_ai/img_0405.jpg'

In [8]:
# Example usage:
image_paths = ['image1.jpg', 'image2.jpg', 'image3.jpg', ...]  # Add all your image paths here
image_crops = []  # Add image crops as OpenCV numpy arrays to this list
target_size = (224, 224)  # Specify your target size
batch_size = 32

image_path_base = [img_path, img_path2]  # Add all your image paths here
image_paths = [ele for ele in image_path_base for i in range(10)]

# Create a CustomDataset instance for image crops
crop_dataset = CustomDataset(image_paths, target_size=target_size)

# Create a DataLoader for batching
data_loader = DataLoader(crop_dataset, batch_size=batch_size, shuffle=False)



# 'feature_embeddings' will contain the normalized feature embeddings for the image crops

# resnet50 60 seconds with 200
# resnet18 59.2 seconds with 200 @ 224x224 = 57.8 secs


In [12]:

# Extract feature embeddings for the image crops with batching
feature_embeddings = extract_resnet_features(data_loader, model=resnet_model)

In [22]:
features_list = []

with torch.no_grad():
    for batch in data_loader:
        features = resnet_model(batch)
    #     features = features.view(features.size(0), -1)
    #     features /= features.norm(p=2, dim=1, keepdim=True)
    #     # features_list.extend(features.detach().cpu().numpy())
    # features_list = features.tolist()

In [24]:
features.shape

torch.Size([20, 512, 1, 1])

In [16]:
len(features[0])

512

In [17]:
features.tolist()

[[0.015534084290266037,
  0.06291692703962326,
  0.0440417118370533,
  0.045794777572155,
  0.029938342049717903,
  0.032636672258377075,
  0.08674739301204681,
  0.007683963049203157,
  0.04891626536846161,
  0.007111322600394487,
  0.0006952866679057479,
  0.011270412243902683,
  0.004873347003012896,
  0.04971972852945328,
  0.12284252047538757,
  0.0025982048828154802,
  0.14172768592834473,
  0.04924238100647926,
  0.04016619175672531,
  0.042598653584718704,
  0.013531922362744808,
  0.0019441109616309404,
  0.027264902368187904,
  0.017779504880309105,
  0.0090066222473979,
  0.028790157288312912,
  0.0019133449532091618,
  0.006860386114567518,
  0.011692467145621777,
  0.028262058272957802,
  0.04721662029623985,
  0.0730927512049675,
  0.0013976816553622484,
  0.0011223017936572433,
  0.0011634696274995804,
  0.005647917743772268,
  0.016675669699907303,
  0.06240342929959297,
  0.022323917597532272,
  0.10492046177387238,
  0.03681822866201401,
  0.023885533213615417,
  0.00

In [ ]:
for batch in data_loader:
    features = model(batch)
    features = features.view(features.size(0), -1)
    features /= features.norm(p=2, dim=1, keepdim=True)

In [31]:
len(feature_embeddings)

200

# From Milvus example

https://milvus.io/docs/integrate_with_pytorch.md

In [25]:
# Load the embedding model with the last layer removed
model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet50', pretrained=True)
model = torch.nn.Sequential(*(list(model.children())[:-1]))
model.eval()

Downloading: "https://github.com/pytorch/vision/zipball/v0.10.0" to /home/superdave/.cache/torch/hub/v0.10.0.zip
/mnt/nvm/repos/ring_detector/.ringenv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/mnt/nvm/repos/ring_detector/.ringenv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Sequential(
  (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (4): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)


In [26]:
# Preprocessing for images
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

https://pytorch.org/rl/tensordict/tutorials/tensorclass_imagenet.html

In [66]:
# Embed function that embeds the batch and inserts it
def embed(data):
    with torch.no_grad():
        output = model(data).squeeze()
    return output

In [42]:
img = Image.open(img_path).convert('RGB')

In [52]:
img_proc = preprocess(img)

In [53]:
img_proc2 = preprocess(img)

In [72]:
batch = torch.stack([img_proc, img_proc2,img_proc, img_proc2,img_proc, img_proc2,img_proc, img_proc2,img_proc, img_proc2,img_proc, img_proc2])

In [68]:
comb = torch.stack([img_proc, img_proc2])
comb.shape

torch.Size([2, 3, 224, 224])

In [73]:

output = embed(batch)

In [75]:
len(output)

12